# Coffee Roast Classification
Notebook mandiri, tidak perlu repository GitHub. Jalankan berurutan dengan **Run all**. Baseline CPU; opsional MobileNet memakai GPU jika tersedia. Dataset gambar dan audio tidak berpasangan, sehingga tidak ada klaim akurasi fusion. Background bukan bukti Green; Medium memerlukan konteks waktu eksternal.

Sumber: [Kaggle](https://www.kaggle.com/datasets/gpiosenka/coffee-bean-dataset-resized-224-x-224), [audio](https://github.com/Theseus-427/coffee-roasting-acoustic-dataset). Internet diperlukan. Unduhan audio memakai Git LFS agar bukan pointer WAV. Jika Kaggle membatasi akses, sel download gagal dengan pesan, bukan melanjutkan dengan data kosong.

In [ ]:
import sys
from pathlib import Path
import os
assert sys.version_info >= (3,12), 'Memerlukan Python 3.12 atau lebih baru'
ROOT = Path('/content/coffee-roast')
ROOT.mkdir(exist_ok=True)
os.chdir(ROOT)


In [ ]:
files = {'coffee/deep.py': '"""Optional frozen MobileNetV3 comparison (ImageNet weights)."""\nimport numpy as np\nfrom PIL import Image\n\n\nclass MobileNetFeatures:\n    def __init__(self):\n        import torch\n        from torchvision.models import MobileNet_V3_Small_Weights, mobilenet_v3_small\n\n        self.torch = torch\n        self.device = torch.device(\'cuda\' if torch.cuda.is_available() else \'cpu\')\n        weights = MobileNet_V3_Small_Weights.DEFAULT\n        self.transform = weights.transforms()\n        self.model = mobilenet_v3_small(weights=weights).to(self.device).eval()\n        self.model.classifier = torch.nn.Identity()\n        probe = torch.zeros(1, 3, 224, 224, device=self.device)\n        with torch.inference_mode():\n            out = self.model(probe)\n        assert out.device == self.device and torch.isfinite(out).all()\n        print(f\'MobileNet: torch={torch.__version__}, device={self.device}, output={out.device}\', flush=True)\n\n    def __call__(self, path):\n        with Image.open(path) as im:\n            tensor = self.transform(im.convert(\'RGB\')).unsqueeze(0).to(self.device)\n        with self.torch.inference_mode():\n            return self.model(tensor).cpu().numpy().reshape(-1).astype(np.float64)\n', 'coffee/download.py': '"""Download public ZIPs, validate archive paths, extract without shell tools."""\nimport argparse\nimport hashlib\nimport json\nimport subprocess\nimport urllib.request\nimport zipfile\nfrom pathlib import Path\n\nSOURCES = {\n    \'image\': (\'https://www.kaggle.com/api/v1/datasets/download/gpiosenka/coffee-bean-dataset-resized-224-x-224?datasetVersionNumber=1\', \'gpiosenka_coffee-bean-dataset-resized-224-x-224\'),\n    \'audio\': (\'https://codeload.github.com/Theseus-427/coffee-roasting-acoustic-dataset/zip/refs/heads/main\', \'.\')}\n\n\ndef safe_extract(archive, target):\n    target = Path(target).resolve()\n    with zipfile.ZipFile(archive) as z:\n        for member in z.infolist():\n            if not (target / member.filename).resolve().is_relative_to(target):\n                raise ValueError(\'Unsafe archive path\')\n        z.extractall(target)\n\n\ndef main():\n    p = argparse.ArgumentParser()\n    p.add_argument(\'--destination\', type=Path, default=Path(\'data\'))\n    p.add_argument(\'--skip-images\', action=\'store_true\', help=\'Use already available local images; only fetch real audio objects\')\n    args = p.parse_args()\n    args.destination.mkdir(parents=True, exist_ok=True)\n    receipts = {}\n    for name, (url, folder) in SOURCES.items():\n        if name == \'image\' and args.skip_images:\n            continue\n        if name == \'audio\':\n            target = args.destination / \'audio-lfs\'\n            subprocess.run([\'git\', \'lfs\', \'version\'], check=True)\n            if not (target / \'.git\').exists():\n                subprocess.run([\'git\', \'clone\', \'https://github.com/Theseus-427/coffee-roasting-acoustic-dataset.git\', str(target)], check=True)\n            subprocess.run([\'git\', \'-C\', str(target), \'lfs\', \'pull\'], check=True)\n            commit = subprocess.check_output([\'git\', \'-C\', str(target), \'rev-parse\', \'HEAD\'], text=True).strip()\n            receipts[name] = {\'repository\': \'https://github.com/Theseus-427/coffee-roasting-acoustic-dataset\', \'commit\': commit}\n            continue\n        archive = args.destination / f\'{name}.zip\'\n        if not archive.exists() or not zipfile.is_zipfile(archive):\n            print(f\'Downloading {name}\', flush=True)\n            request = urllib.request.Request(url, headers={\'User-Agent\': \'coffee-roast-research/1.0\'})\n            with urllib.request.urlopen(request, timeout=120) as response, archive.open(\'wb\') as f:\n                while chunk := response.read(1024*1024):\n                    f.write(chunk)\n        if not zipfile.is_zipfile(archive):\n            raise RuntimeError(f\'{name}: server did not return ZIP; check dataset access at {url}\')\n        safe_extract(archive, args.destination / folder)\n        receipts[name] = {\'url\': url, \'sha256\': hashlib.sha256(archive.read_bytes()).hexdigest()}\n    (args.destination / \'download_receipts.json\').write_text(json.dumps(receipts, indent=2), encoding=\'utf-8\')\n\n\nif __name__ == \'__main__\':\n    main()\n', 'coffee/features.py': "from pathlib import Path\n\nimport numpy as np\nfrom PIL import Image\nfrom scipy.io import wavfile\nfrom scipy.signal import find_peaks\n\n\ndef image_features(path):\n    with Image.open(path) as im:\n        rgb = im.convert('RGB').resize((224, 224))\n        arrays = [np.asarray(rgb) / 255., np.asarray(rgb.convert('HSV')) / 255.]\n    values = []\n    for array in arrays:\n        for channel in array.transpose(2, 0, 1):\n            hist = np.histogram(channel, bins=16, range=(0, 1))[0]\n            values.extend(hist / hist.sum())\n            values.extend([channel.mean(), channel.std()])\n    return np.asarray(values, dtype=np.float64)\n\n\ndef audio_features(path):\n    with open(path, 'rb') as f:\n        if f.read(7) == b'version':\n            raise ValueError(f'{path}: Git LFS pointer, run python -m coffee.download and use data/audio-lfs/datasets')\n    sr, raw = wavfile.read(Path(path))\n    if raw.size == 0:\n        raise ValueError('Empty audio')\n    x = raw.astype(np.float64)\n    if np.issubdtype(raw.dtype, np.integer):\n        info = np.iinfo(raw.dtype)\n        if raw.dtype == np.uint8:\n            x = (x - 128) / 128\n        else:\n            x /= max(abs(info.min), info.max)\n    if x.ndim == 2:\n        x = x.mean(axis=1)\n    duration = len(x) / sr\n    peak = np.abs(x).max()\n    rms = np.sqrt(np.mean(x ** 2))\n    power = np.abs(np.fft.rfft(x * np.hanning(len(x)))) ** 2\n    freq = np.fft.rfftfreq(len(x), 1 / sr)\n    total = power.sum() + 1e-20\n    centroid = np.sum(freq * power) / total\n    bandwidth = np.sqrt(np.sum((freq - centroid) ** 2 * power) / total)\n    rolloff = freq[min(np.searchsorted(np.cumsum(power), .85 * total), len(freq)-1)]\n    bands = [power[(freq >= lo) & (freq < hi)].sum() / total\n             for lo, hi in [(0, 2000), (2000, 4000), (4000, 8000), (8000, 12000), (12000, sr/2+1)]]\n    # Local transient density, NOT true crack rate of the full roasting trial.\n    peaks, _ = find_peaks(np.abs(x), height=max(rms * 3, 1e-9), distance=max(1, int(sr*.01)))\n    gaps = np.diff(peaks) / sr\n    return np.asarray([duration, peak, rms, np.mean(x[:-1]*x[1:] < 0) if len(x)>1 else 0,\n                       centroid/sr, bandwidth/sr, rolloff/sr, *bands,\n                       len(peaks)/duration, gaps.mean() if gaps.size else 0,\n                       gaps.std() if gaps.size else 0], dtype=np.float64)\n", 'coffee/fusion.py': 'import numpy as np\n\nROASTS = [\'Green\', \'Light\', \'Medium\', \'Dark\']\n\n\ndef fuse(image, audio, image_accuracy, audio_accuracy, *, background_stage=None):\n    """Experimental heuristic; background needs externally observed roast context.\n\n    background_stage: Green before first crack, Medium between cracks.\n    Without context, audio abstains for Background and image evidence is retained.\n    """\n    image, audio = np.asarray(image, float), np.asarray(audio, float)\n    for p, size in [(image, 4), (audio, 3)]:\n        if p.shape != (size,) or not np.isfinite(p).all() or (p < 0).any() or not np.isclose(p.sum(), 1):\n            raise ValueError(\'Expected normalized probability vector\')\n    if background_stage not in (None, \'Green\', \'Medium\'):\n        raise ValueError(\'background_stage must be Green, Medium, or None\')\n    if not all(np.isfinite(a) and 0 <= a <= 1 for a in (image_accuracy, audio_accuracy)) or image_accuracy+audio_accuracy == 0:\n        raise ValueError(\'Invalid validation accuracies\')\n    mapped = np.array([0., audio[1], 0., audio[2]])\n    if background_stage:\n        mapped[ROASTS.index(background_stage)] = audio[0]\n    else:\n        mapped += audio[0] * image\n    return (image_accuracy * image + audio_accuracy * mapped) / (image_accuracy + audio_accuracy)\n', 'coffee/predict.py': "import argparse\nimport json\nfrom pathlib import Path\n\nimport joblib\n\nfrom .features import audio_features, image_features\nfrom .fusion import ROASTS, fuse\n\n\ndef main():\n    p = argparse.ArgumentParser()\n    p.add_argument('--image', required=True)\n    p.add_argument('--audio', required=True)\n    p.add_argument('--models', type=Path, default=Path('outputs'))\n    p.add_argument('--background-stage', choices=['Green', 'Medium'])\n    args = p.parse_args()\n    image = joblib.load(args.models / 'image.joblib')\n    audio = joblib.load(args.models / 'audio.joblib')\n    pi = image['model'].predict_proba([image_features(args.image)])[0]\n    pa = audio['model'].predict_proba([audio_features(args.audio)])[0]\n    result = fuse(pi, pa, image['validation_accuracy'], audio['validation_accuracy'], background_stage=args.background_stage)\n    print(json.dumps({'image': dict(zip(ROASTS, pi)), 'audio': dict(zip(audio['classes'], pa)),\n                      'experimental_fusion': dict(zip(ROASTS, result)), 'prediction': ROASTS[result.argmax()]}, indent=2))\n\n\nif __name__ == '__main__':\n    main()\n", 'coffee/train.py': '"""Run: python -m coffee.train --output outputs"""\nimport argparse\nimport hashlib\nimport json\nimport re\nfrom collections import Counter\nfrom pathlib import Path\n\nimport joblib\nimport numpy as np\nfrom sklearn.ensemble import RandomForestClassifier\nfrom sklearn.metrics import accuracy_score, classification_report, confusion_matrix\nfrom sklearn.model_selection import train_test_split\nfrom sklearn.pipeline import make_pipeline\nfrom sklearn.preprocessing import StandardScaler\nfrom sklearn.svm import SVC\n\nfrom .features import audio_features, image_features\nfrom .fusion import ROASTS\n\n\ndef discover(root, classes, suffix):\n    rows = []\n    for split in [\'train\', \'val\', \'test\']:\n        for label, name in enumerate(classes):\n            for p in sorted((root / split / name).glob(\'*\')):\n                if p.suffix.lower() in suffix:\n                    rows.append({\'path\': str(p), \'label\': label, \'split\': split})\n    if not rows:\n        raise ValueError(f\'No dataset at {root}\')\n    return rows\n\n\ndef deduplicate(rows):\n    # Keep held-out data; remove duplicate copies from lower-priority splits.\n    seen = {}\n    unique = []\n    priority = {\'test\': 0, \'val\': 1, \'train\': 2}\n    for row in sorted(rows, key=lambda r: priority[r[\'split\']]):\n        digest = hashlib.sha256(Path(row[\'path\']).read_bytes()).hexdigest()\n        if digest in seen:\n            if seen[digest] != row[\'label\']:\n                raise ValueError(\'Identical file has conflicting labels\')\n            continue\n        seen[digest] = row[\'label\']\n        unique.append(row)\n    return unique\n\n\ndef image_rows(root):\n    rows = deduplicate(discover(root, ROASTS, {\'.png\', \'.jpg\', \'.jpeg\'}))\n    # Reserve 15% of deduplicated total from the remaining train pool.\n    train = [r for r in rows if r[\'split\'] == \'train\']\n    _, val_indices = train_test_split(np.arange(len(train)), test_size=.15*len(rows)/len(train),\n                                     stratify=[r[\'label\'] for r in train], random_state=42)\n    for i in val_indices:\n        train[i][\'split\'] = \'val\'\n    return rows\n\n\ndef audit(rows):\n    hashes = {}\n    groups = {s: set() for s in [\'train\', \'val\', \'test\']}\n    duplicates = 0\n    for row in rows:\n        digest = hashlib.sha256(Path(row[\'path\']).read_bytes()).hexdigest()\n        if digest in hashes and hashes[digest] != row[\'split\']:\n            duplicates += 1\n        hashes[digest] = row[\'split\']\n        match = re.match(r\'([^_]+)_\', Path(row[\'path\']).stem)\n        if match:\n            groups[row[\'split\']].add(match[1])\n    return {\'counts\': dict(Counter(r[\'split\'] for r in rows)),\n            \'cross_split_identical_files\': duplicates,\n            \'train_test_trial_overlap\': sorted(groups[\'train\'] & groups[\'test\'])}\n\n\ndef run_modality(rows, feature_fn, names, output, name):\n    output.mkdir(parents=True, exist_ok=True)\n    (output / f\'{name}_manifest.json\').write_text(json.dumps(rows, indent=2), encoding=\'utf-8\')\n    features = np.stack([feature_fn(r[\'path\']) for r in rows])\n    if not np.isfinite(features).all():\n        raise ValueError(\'Non-finite features\')\n    labels = np.array([r[\'label\'] for r in rows])\n    masks = {s: np.array([r[\'split\'] == s for r in rows]) for s in [\'train\', \'val\', \'test\']}\n    for s, mask in masks.items():\n        if set(labels[mask]) != set(range(len(names))):\n            raise ValueError(f\'{name}/{s}: missing classes\')\n    models = {\'svm\': make_pipeline(StandardScaler(), SVC(probability=True, random_state=42)),\n              \'rf\': RandomForestClassifier(n_estimators=200, class_weight=\'balanced\', random_state=42, n_jobs=-1)}\n    val_scores = {}\n    for key, model in models.items():\n        model.fit(features[masks[\'train\']], labels[masks[\'train\']])\n        val_scores[key] = accuracy_score(labels[masks[\'val\']], model.predict(features[masks[\'val\']]))\n    selected = max(val_scores, key=val_scores.get)\n    model = models[selected]\n    # Test is used once, only after selecting by validation accuracy.\n    pred = model.predict(features[masks[\'test\']])\n    report = {\'audit\': audit(rows), \'selected\': selected, \'validation_accuracy\': val_scores,\n              \'test_accuracy\': accuracy_score(labels[masks[\'test\']], pred),\n              \'test_report\': classification_report(labels[masks[\'test\']], pred, target_names=names, output_dict=True, zero_division=0),\n              \'confusion_matrix\': confusion_matrix(labels[masks[\'test\']], pred).tolist()}\n    joblib.dump({\'model\': model, \'classes\': names, \'validation_accuracy\': val_scores[selected]}, output / f\'{name}.joblib\')\n    np.savez_compressed(output / f\'{name}_test_predictions.npz\', labels=labels[masks[\'test\']],\n                        probabilities=model.predict_proba(features[masks[\'test\']]))\n    return report\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\'--images\', type=Path, default=Path(\'gpiosenka_coffee-bean-dataset-resized-224-x-224\'))\n    parser.add_argument(\'--audio\', type=Path, default=Path(\'data/audio-lfs/datasets\'))\n    parser.add_argument(\'--output\', type=Path, default=Path(\'outputs\'))\n    parser.add_argument(\'--cnn\', action=\'store_true\', help=\'Also compare frozen ImageNet MobileNetV3 features; requires torch/torchvision\')\n    args = parser.parse_args()\n    results = {}\n    for name, rows, fn, classes in [\n        (\'image\', image_rows(args.images), image_features, ROASTS),\n        (\'audio\', deduplicate(discover(args.audio, [\'0\', \'1\', \'2\'], {\'.wav\'})), audio_features, [\'Background\', \'First Crack\', \'Second Crack\'])]:\n        print(f\'{name}: extracting {len(rows)} files\', flush=True)\n        results[name] = run_modality(rows, fn, classes, args.output, name)\n        print(f\'{name}: test accuracy={results[name]["test_accuracy"]:.4f}\', flush=True)\n    results[\'fusion\'] = \'Not evaluated: no paired ground truth. Validation weights are heuristic across different tasks.\'\n    if args.cnn:\n        from .deep import MobileNetFeatures\n        results[\'image_cnn\'] = run_modality(image_rows(args.images), MobileNetFeatures(), ROASTS, args.output, \'image_cnn\')\n    (args.output / \'metrics.json\').write_text(json.dumps(results, indent=2), encoding=\'utf-8\')\n\n\nif __name__ == \'__main__\':\n    main()\n', 'coffee/__init__.py': '"""Coffee roast classification with independent image and audio baselines."""\n', 'tests/test_project.py': "import io\nimport zipfile\nfrom pathlib import Path\n\nimport numpy as np\nimport pytest\nfrom PIL import Image\nfrom scipy.io import wavfile\n\nfrom coffee.download import safe_extract\nfrom coffee.features import audio_features, image_features\nfrom coffee.fusion import fuse\nfrom coffee.train import audit, deduplicate, discover, image_rows\n\n\ndef test_image_features(tmp_path):\n    path = tmp_path / 'red.png'\n    Image.new('RGB', (224, 224), (255, 0, 0)).save(path)\n    result = image_features(path)\n    assert result.shape == (108,)\n    assert np.isfinite(result).all()\n    assert result[16] == 1\n\n\ndef test_audio_features(tmp_path):\n    path = tmp_path / 'silence.wav'\n    wavfile.write(path, 44100, np.zeros(4410, np.int16))\n    result = audio_features(path)\n    assert result.shape == (15,)\n    assert np.isfinite(result).all()\n    assert result[0] == .1\n    assert result[1] == 0\n\n\ndef test_fusion():\n    image = np.array([.1, .2, .6, .1])\n    np.testing.assert_allclose(fuse(image, [1, 0, 0], .8, .9), image)\n    result = fuse(image, [1, 0, 0], .8, .9, background_stage='Medium')\n    assert result.argmax() == 2\n    assert result.sum() == pytest.approx(1)\n    assert fuse(image, [0, 0, 1], .8, .9).argmax() == 3\n    with pytest.raises(ValueError):\n        fuse(image, [1, 1, 1], .8, .9)\n\n\ndef test_zip_traversal(tmp_path):\n    data = io.BytesIO()\n    with zipfile.ZipFile(data, 'w') as z:\n        z.writestr('../escape.txt', 'bad')\n    data.seek(0)\n    with pytest.raises(ValueError):\n        safe_extract(data, tmp_path)\n\n\ndef test_lfs_pointer_rejected(tmp_path):\n    path = tmp_path / 'pointer.wav'\n    path.write_text('version https://git-lfs.github.com/spec/v1\\n', encoding='utf-8')\n    with pytest.raises(ValueError, match='Git LFS pointer'):\n        audio_features(path)\n\n\ndef test_zip_extract(tmp_path):\n    data = io.BytesIO()\n    with zipfile.ZipFile(data, 'w') as z:\n        z.writestr('nested/sample.txt', 'coffee')\n    data.seek(0)\n    safe_extract(data, tmp_path)\n    assert (tmp_path / 'nested/sample.txt').read_text() == 'coffee'\n\n\ndef test_local_datasets():\n    root = Path('gpiosenka_coffee-bean-dataset-resized-224-x-224')\n    if not root.exists():\n        root = Path('data/gpiosenka_coffee-bean-dataset-resized-224-x-224')\n    audio = Path('data/audio-lfs/datasets')\n    if not root.exists() or not audio.exists():\n        pytest.skip('Local datasets absent; feature and fusion tests still run')\n    rows = image_rows(root)\n    assert len(rows) == 1194  # 1600 source files minus 406 duplicate copies.\n    assert rows == image_rows(root)\n    assert len({r['path'] for r in rows}) == len(rows)\n    assert audit(rows)['cross_split_identical_files'] == 0\n    for split in ['train', 'val', 'test']:\n        assert {r['label'] for r in rows if r['split'] == split} == {0, 1, 2, 3}\n    sounds = discover(audio, ['0', '1', '2'], {'.wav'})\n    assert len(sounds) == 4304  # Actual upstream files; README states 4301.\n    assert audit(deduplicate(sounds))['cross_split_identical_files'] == 0\n    assert np.isfinite(audio_features(sounds[0]['path'])).all()\n    assert np.isfinite(image_features(rows[0]['path'])).all()\n", 'requirements.txt': 'numpy==2.5.4\nscipy==1.18.1\nscikit-learn==1.9.1\nPillow==13.0.0\njoblib==1.6.0\npytest==9.2.0\n'}
for name, content in files.items():
    path = Path(name)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(content, encoding='utf-8')


In [ ]:
%pip install -q -r requirements.txt
!apt-get -qq update
!apt-get -qq install -y git-lfs
import subprocess
subprocess.run(['git','lfs','install'],check=True)


In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'coffee.download', '--destination', 'data'], check=True)


In [ ]:
subprocess.run([sys.executable,'-m','pytest','-q'],check=True)
# False: baseline yang diuji lokal; True: tambah perbandingan CNN.
RUN_CNN = False
if RUN_CNN:
    subprocess.run([sys.executable,'-m','pip','install','torch','torchvision'],check=True)
command = [sys.executable,'-m','coffee.train','--images','data/gpiosenka_coffee-bean-dataset-resized-224-x-224','--audio','data/audio-lfs/datasets']
if RUN_CNN:
    command.append('--cnn')
subprocess.run(command,check=True)


In [ ]:
import json
metrics = json.loads(Path('outputs/metrics.json').read_text())
for modality, result in metrics.items():
    if isinstance(result, dict):
        print(modality, 'selected:',result['selected'], 'test accuracy:',result['test_accuracy'])
        print('Audit:',result['audit'])
print(metrics['fusion'])


## Inferensi fusion opsional
Gunakan gambar dan WAV dari observasi roasting yang sama. Pasangan acak lintas dataset hanya menguji fungsi, bukan validasi multimodal. `--background-stage Medium` hanya bila Anda mengetahui observasi berada di antara first dan second crack. Tanpa konteks, massa probabilitas Background mengikuti gambar (audio abstain).

In [ ]:
# Contoh setelah upload input milik Anda:
# subprocess.run([sys.executable,'-m','coffee.predict','--image','bean.png','--audio','event.wav'],check=True)
import shutil
from google.colab import files
shutil.make_archive('/content/coffee-results','zip','outputs')
files.download('/content/coffee-results.zip')
